# VentIA — experimento preliminar\nEste cuaderno reproduce la comparación de modelos descrita en el informe. Coloca el ZIP del dataset en la ruta indicada.\nLa serie objetivo es ventas totales diarias agregadas; el último bloque de 90 días se reserva para prueba.

In [ ]:

# VentIA — experimento reproducible de pronóstico
# Requisitos: pandas, numpy, scikit-learn, statsmodels, torch, matplotlib
import zipfile, pandas as pd, numpy as np, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ZIP_PATH = "store-sales-time-series-forecasting.zip"  # ajustar ruta local
z = zipfile.ZipFile(ZIP_PATH)
df = pd.read_csv(z.open("train.csv"), usecols=["date","sales"], parse_dates=["date"])
s = df.groupby("date")["sales"].sum().sort_index().asfreq("D").fillna(0)
H = 90
y = s.values.astype(float); ytr, yte = y[:-H], y[-H:]; dates=s.index[-H:]
pred = {"Naive estacional (7 días)": np.array([ytr[-7+i%7] for i in range(H)])}
pred["Holt-Winters"] = ExponentialSmoothing(ytr, trend="add", seasonal="add",
    seasonal_periods=7, initialization_method="estimated").fit().forecast(H)

def feat(hist, dt):
    h=np.asarray(hist)
    return [h[-1],h[-7],h[-14],h[-28],np.mean(h[-7:]),np.mean(h[-28:]),dt.dayofweek,dt.month]
X=[]; Y=[]
for i in range(28,len(ytr)):
    X.append(feat(ytr[:i],s.index[i])); Y.append(ytr[i])
X=np.asarray(X); Y=np.asarray(Y)
models={"Árbol de decisión":DecisionTreeRegressor(max_depth=8,min_samples_leaf=5,random_state=42),
        "Random Forest":RandomForestRegressor(n_estimators=100,max_depth=12,min_samples_leaf=3,n_jobs=-1,random_state=42)}
for name,m in models.items():
    m.fit(X,Y); hist=list(ytr); p=[]
    for dt in dates:
        q=max(0,float(m.predict([feat(hist,dt)])[0])); p.append(q); hist.append(q)
    pred[name]=np.array(p)

# GRU univariada, 28 pasos de entrada y pronóstico recursivo.
torch.manual_seed(42); np.random.seed(42)
log=np.log1p(ytr); mu,sd=log.mean(),log.std(); zsc=(log-mu)/sd; L=28
xx=np.array([zsc[i-L:i] for i in range(L,len(zsc))],dtype=np.float32)
yy=np.array([zsc[i] for i in range(L,len(zsc))],dtype=np.float32)
loader=DataLoader(TensorDataset(torch.tensor(xx).unsqueeze(-1),torch.tensor(yy).unsqueeze(-1)),batch_size=64,shuffle=True)
class GRUForecast(nn.Module):
    def __init__(self):
        super().__init__(); self.gru=nn.GRU(1,32,batch_first=True); self.fc=nn.Linear(32,1)
    def forward(self,x):
        o,_=self.gru(x); return self.fc(o[:,-1,:])
net=GRUForecast(); opt=torch.optim.Adam(net.parameters(),lr=.003); lossfn=nn.MSELoss()
for epoch in range(35):
    for xb,yb in loader:
        opt.zero_grad(); loss=lossfn(net(xb),yb); loss.backward(); opt.step()
hist=list(zsc); p=[]; net.eval()
with torch.no_grad():
    for _ in range(H):
        q=float(net(torch.tensor(hist[-L:],dtype=torch.float32).view(1,L,1)).item())
        p.append(max(0,float(np.expm1(q*sd+mu)))); hist.append(q)
pred["GRU"]=np.array(p)

def metric(a,p):
    a=np.maximum(0,a); p=np.maximum(0,p); nz=a!=0
    return [mean_absolute_error(a,p),np.sqrt(mean_squared_error(a,p)),
            np.sqrt(mean_squared_log_error(a,p)),np.mean(np.abs((a[nz]-p[nz])/a[nz]))*100]
results=pd.DataFrame({k:metric(yte,v) for k,v in pred.items()},
    index=["MAE","RMSE","RMSLE","MAPE (%)"]).T.sort_values("RMSLE")
display(results)
